# HW02 — Download the hourly Wikipedia data

Download the course's hourly Parquet extracts to EC2. These temporary files are excluded from Git by `.gitignore`.


In [1]:
from pathlib import Path

import boto3
from botocore import UNSIGNED
from botocore.config import Config

SOURCE_BUCKET = "dsan6000-wikipedia"
SOURCE_PREFIX = "hourly_parquet/"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# The source bucket is public; no credentials are embedded in this notebook.
source_s3 = boto3.client("s3", region_name="us-east-1", config=Config(signature_version=UNSIGNED))

In [2]:
pages = source_s3.get_paginator("list_objects_v2").paginate(
    Bucket=SOURCE_BUCKET, Prefix=SOURCE_PREFIX
)
objects = sorted(
    [obj for page in pages for obj in page.get("Contents", [])
     if obj["Key"].endswith(".parquet")],
    key=lambda obj: obj["Key"],
)
assert len(objects) == 24, f"Expected 24 hourly files, found {len(objects)}. Inspect the source listing."
assert len({Path(obj["Key"]).name for obj in objects}) == len(objects)
print(f"Found {len(objects)} hourly Parquet files.")
for obj in objects:
    print(obj["Key"], obj["Size"], "bytes")

Found 24 hourly Parquet files.
hourly_parquet/20260901_040000.parquet 1153958 bytes
hourly_parquet/20260901_050000.parquet 1029131 bytes
hourly_parquet/20260901_060000.parquet 1039577 bytes
hourly_parquet/20260901_070000.parquet 867454 bytes
hourly_parquet/20260901_080000.parquet 953254 bytes
hourly_parquet/20260901_090000.parquet 916922 bytes
hourly_parquet/20260901_100000.parquet 1006317 bytes
hourly_parquet/20260901_110000.parquet 1123903 bytes
hourly_parquet/20260901_120000.parquet 1252861 bytes
hourly_parquet/20260901_130000.parquet 1631521 bytes
hourly_parquet/20260901_140000.parquet 1619419 bytes
hourly_parquet/20260901_150000.parquet 1487553 bytes
hourly_parquet/20260901_160000.parquet 1576244 bytes
hourly_parquet/20260901_170000.parquet 1560475 bytes
hourly_parquet/20260901_180000.parquet 1587132 bytes
hourly_parquet/20260901_190000.parquet 1515010 bytes
hourly_parquet/20260901_200000.parquet 1848643 bytes
hourly_parquet/20260901_210000.parquet 1787572 bytes
hourly_parquet/202

In [3]:
for obj in objects:
    destination = DATA_DIR / Path(obj["Key"]).name
    source_s3.download_file(SOURCE_BUCKET, obj["Key"], str(destination))
    assert destination.stat().st_size == obj["Size"], f"Size mismatch: {destination.name}"

print(f"Downloaded and verified {len(objects)} files.")
print(f"Total size: {sum(obj['Size'] for obj in objects):,} bytes")

Downloaded and verified 24 files.
Total size: 32,012,082 bytes
